# 10 Full Run Readiness

目的: smoke から本格実行へ移る前に、Drive artifact、GPU/CUDA、download gate、次に足りない artifact を確認します。

この notebook は安全確認用です。デフォルトでは大規模 download、model download、training は実行しません。

固定 path:

- code root: `/content/drive/MyDrive/codex/batting_codex_handoff`
- artifact root: `/content/drive/MyDrive/baseball_vision`
- cache root: `/content/cache/baseball_vision`

次に狙う流れ:

1. real Statcast input で `11_download_statcast_and_video_sources.ipynb`
2. `11_download_statcast_and_video_sources.ipynb`
3. direct `media_url` がある source だけ明示 download
4. `clips_v1` を作る full CV preprocessing
5. structured sequence / player-season prior
6. frozen video/image encoder baseline
7. late fusion
8. `09_report_builder.ipynb`


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
CONTEXT_RUN_ID = run_id(RUN_PROFILE, 'context_run_id')
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('CONTEXT_RUN_ID =', CONTEXT_RUN_ID)
print('FULL_RUN_ID =', FULL_RUN_ID)


## Readiness Check

CPU でよい stage と GPU 必須 stage を分けて表示します。`REQUIRE_GPU=True` にすると、CUDA が使えない runtime では blocker として扱います。

In [ ]:
from sport_pipeline.pipeline.full_run_preflight import build_full_run_readiness_report

REQUIRE_GPU = False
readiness_path = BASE_DIR / 'reports' / 'preflight' / f'full_run_readiness_{FULL_RUN_ID}.json'

report = build_full_run_readiness_report(
    base_dir=BASE_DIR,
    cache_dir=CACHE_DIR,
    repo_dir=REPO_DIR,
    config_path=RUN_PROFILE_PATH,
    context_run_id=CONTEXT_RUN_ID,
    full_run_id=FULL_RUN_ID,
    require_gpu=REQUIRE_GPU,
    output_json=readiness_path,
)

print('readiness json ->', readiness_path)
print('selected_device =', report['device']['selected_device'])
print('cuda_available =', report['device']['cuda_available'])
print('gpu_name =', report['device']['gpu_name'])
print('gpu_total_memory_gb =', report['device']['gpu_total_memory_gb'])
print('statcast_date_range =', report.get('statcast_date_range'))

if report['blockers']:
    print('\nBLOCKERS:')
    for item in report['blockers']:
        print('-', item)
else:
    print('\nNo required blockers for the currently implemented full-run gate.')

if report['warnings']:
    print('\nWARNINGS / NEXT MISSING OPTIONAL ARTIFACTS:')
    for item in report['warnings']:
        print('-', item)

print('\nArtifact groups:')
for group in report['artifact_groups']:
    print(group['name'], 'required=', group['required'], 'all_present=', group['all_present'])

if report.get('semantic_checks'):
    print('\nSemantic checks:')
    for item in report['semantic_checks']:
        print(item['name'], {k: v for k, v in item.items() if k not in {'name', 'path'}})


## Optional Direct Media Download

`video_sources_v1.parquet` に direct `media_url` が入っている row だけを対象にします。ページ scraping や access bypass はしません。

最初は `ENABLE_DIRECT_MEDIA_DOWNLOAD=False` のまま plan だけ見てください。実行する場合だけ `True` に変更します。

In [ ]:
from sport_pipeline.video.download_sources import download_video_sources

DOWNLOAD_SETTINGS = stage_settings(RUN_PROFILE, 'video_download')
# 10 is a readiness notebook. Keep this False even when the real profile enables downloads; run notebook 11 for execution.
ENABLE_DIRECT_MEDIA_DOWNLOAD = False
MAX_FILES = DOWNLOAD_SETTINGS.get('max_files')
MAX_BYTES_PER_FILE = DOWNLOAD_SETTINGS.get('max_bytes_per_file')
TIMEOUT_SEC = int(DOWNLOAD_SETTINGS.get('timeout_sec', 900))
ALLOW_HLS = bool(DOWNLOAD_SETTINGS.get('allow_hls', True))

video_sources = BASE_DIR / 'manifests' / 'video_sources_v1.parquet'
download_dir = BASE_DIR / 'raw_videos' / FULL_RUN_ID
# Readiness must not overwrite the real raw_videos download manifest. Notebook 11 owns that artifact.
download_plan_manifest = BASE_DIR / 'reports' / 'preflight' / f'direct_media_download_plan_{FULL_RUN_ID}.parquet'

if not video_sources.exists():
    raise FileNotFoundError(f'video_sources_v1 がありません。先に notebooks/11_download_statcast_and_video_sources.ipynb を実行してください: {video_sources}')

rows = download_video_sources(
    video_sources_path=video_sources,
    output_dir=download_dir,
    output_manifest=download_plan_manifest,
    execute=ENABLE_DIRECT_MEDIA_DOWNLOAD,
    max_files=MAX_FILES,
    max_bytes_per_file=MAX_BYTES_PER_FILE,
    timeout_sec=TIMEOUT_SEC,
    allow_hls=ALLOW_HLS,
)

downloaded = [row for row in rows if row.get('download_status') == 'downloaded']
print('download_plan_manifest ->', download_plan_manifest)
print('execute =', ENABLE_DIRECT_MEDIA_DOWNLOAD)
print('max_files =', MAX_FILES)
print('rows =', len(rows), 'downloaded =', len(downloaded))
for row in rows[:20]:
    print(row['video_source_id'], row['download_status'], row.get('planned_path'), row.get('skip_reason') or row.get('error'))


## Next Human Actions

- CPU でよい: `00`, `01`, `02`, `03`, `05`, `09`, この `10`
- GPU 推奨: full CV preprocessing、structured sequence training、video/image encoder baseline
- 初回 paid runtime 推奨: L4
- 大きめの VideoMAE / 長い clip / batch を増やす場合: A100

`clips/{FULL_RUN_ID}/clips_v1.parquet` が無い間は、sequence / video baseline / fusion は本格 artifact ではなく smoke 段階です。